# 02 - Data cleaning
**Project:** E-commerce Sales Dashboard (Online Retail dataset)

**Goal of this notebook:** turn the raw data into a clean table that answers our business questions. Every step prints the number of rows left, so we can report exactly what was removed and why.

**Input:** `../data/rawData1.csv`  |  **Output:** `../data/cleanData.csv`

> Run the notebook from top to bottom. Use **Restart and Run All** before pushing it to GitHub.

## 1. Load the raw data

In [1]:
import os
import pandas as pd

df = pd.read_csv("../data/rawData1.csv")
print("Raw data:", df.shape)

Raw data: (541909, 8)


## 2. Cleaning steps
We work on a **copy** (`df_clean`) so the raw data stays untouched. The function `log_step` records the number of rows after each step, so we can build a summary table at the end.

In [3]:
df_clean = df.copy()
steps = [("Start", len(df_clean))]

def log_step(name, table):
    steps.append((name, len(table)))
    print(f"{name}: {len(table):,} rows")

log_step("Raw data", df_clean)

Raw data: 541,909 rows


### Step 1 - Keep only numeric invoice numbers
Invoice numbers starting with `C` are cancellations (we verified that in the exploration notebook). A few other invoices contain a letter too. None of them is a real sale, so we keep only invoices made of digits.

In [4]:
df_clean = df_clean[df_clean["InvoiceNo"].str.isdigit()]
log_step("After keeping numeric invoices only", df_clean)

After keeping numeric invoices only: 532,618 rows


### Step 2 - Remove zero or negative quantities
After step 1, this removes the negative quantities that were not cancellations (returns and stock adjustments).

In [5]:
df_clean = df_clean[df_clean["Quantity"] > 0]
log_step("After removing quantity <= 0", df_clean)

After removing quantity <= 0: 531,282 rows


### Step 3 - Remove zero or negative prices
A price of zero or less is not a real sale (free items, corrections, and bad debt adjustments).

In [6]:
df_clean = df_clean[df_clean["UnitPrice"] > 0]
log_step("After removing price <= 0", df_clean)

After removing price <= 0: 530,103 rows


### Step 4 - Remove duplicate rows
`drop_duplicates()` keeps the first copy of each identical row.

In [7]:
df_clean = df_clean.drop_duplicates()
log_step("After removing duplicates", df_clean)

After removing duplicates: 524,877 rows


## 3. Fix types and create new columns
- **Product names:** remove extra spaces and use capitals, so the same product is not counted twice.
- **`InvoiceDate`:** convert from text to a real date.
- **`Revenue`:** quantity x unit price.
- **`Month`:** the date written as `2011-03`, easy to group by and to read in Power BI.

In [8]:
df_clean["Description"] = df_clean["Description"].str.strip().str.upper()
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]
df_clean["Month"] = df_clean["InvoiceDate"].dt.strftime("%Y-%m")

df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Month
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12


## 4. Specific tables for specific questions
We do **not** delete rows from `df_clean` for these two cases, because that would remove real sales from the other questions. We make separate tables instead.

| Table | Content | Used for |
|---|---|---|
| `df_clean` | all valid sales | months, countries, KPIs |
| `customers_only` | rows with a known `CustomerID` | top customers, number of customers |
| `products_only` | rows whose `StockCode` starts with 5 digits | top products (excludes `POST`, `DOT`, and other fees) |

In [9]:
customers_only = df_clean.dropna(subset=["CustomerID"])
products_only = df_clean[df_clean["StockCode"].str.match(r"^\d{5}")]

print("Rows with a known customer:", len(customers_only))
print("Rows excluded from the product ranking:", len(df_clean) - len(products_only))

Rows with a known customer: 392692
Rows excluded from the product ranking: 2373


## 5. Cleaning summary (for the report)

In [10]:
summary = pd.DataFrame(steps, columns=["Step", "Rows left"])
summary["Rows removed"] = (summary["Rows left"].shift(1) - summary["Rows left"]).fillna(0).astype(int)
summary = summary.iloc[1:].reset_index(drop=True)   # drop the duplicate 'Start' line
summary

,Step,Rows left,Rows removed
0,Raw data,541909,0
1,After keeping numeric invoices only,532618,9291
2,After removing quantity <= 0,531282,1336
3,After removing price <= 0,530103,1179
4,After removing duplicates,524877,5226


In [11]:
total_removed = len(df) - len(df_clean)
print(f"Total removed: {total_removed:,} rows ({total_removed / len(df):.1%} of the raw data)")

Total removed: 17,032 rows (3.1% of the raw data)


## 6. Final checks
If one of these fails, a cleaning step did not work.

In [12]:
assert df_clean["Quantity"].min() > 0
assert df_clean["UnitPrice"].min() > 0
assert df_clean["InvoiceNo"].str.isdigit().all()
assert df_clean.duplicated().sum() == 0

print("Date range:", df_clean["InvoiceDate"].min(), "to", df_clean["InvoiceDate"].max())
print("Missing values per column:")
print(df_clean.isna().sum())
print()
df_clean.info()

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Missing values per column:
InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132185
Country             0
Revenue             0
Month               0
dtype: int64

<class 'pandas.DataFrame'>
Index: 524877 entries, 0 to 541908
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    524877 non-null  str           
 1   StockCode    524877 non-null  str           
 2   Description  524877 non-null  str           
 3   Quantity     524877 non-null  int64         
 4   InvoiceDate  524877 non-null  datetime64[us]
 5   UnitPrice    524877 non-null  float64       
 6   CustomerID   392692 non-null  float64       
 7   Country      524877 non-null  str           
 8   Revenue      524877 non-null  float64       
 9   Month        524877 non-null  str 

> **Reminder:** December 2011 is incomplete (the data stops on 9 December), so its revenue will look low. This is not an error, and we will mention it in the dashboard and the report.

## 7. Save the clean data
This file is the input for the analysis notebook and for Power BI.

In [13]:
os.makedirs("../data", exist_ok=True)
df_clean.to_csv("../data/cleanData.csv", index=False)
print("Saved:", os.path.abspath("../data/cleanData.csv"))

Saved: C:\Users\wadie\Desktop\PROJET DATA\data\cleanData.csv


**Next step:** `03_analysis.ipynb`, where we answer the business questions (best products, revenue by month, and the third question).